In [ ]:

# DeepHF: Prediction and SHAP Explainability

This notebook loads 23-nt sequences, extracts sequence and biological features, trains a BiLSTM-based DeepHF model, and performs SHAP analysis for interpretability. Output includes:

- Predicted DeepHF efficiencies
- Engineered feature values
- SHAP values
- Waterfall plots


In [ ]:
# Imports and Environment Setup in linux WSL abuntu

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import shap
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Concatenate, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from Bio.SeqUtils import MeltingTemp as Tm
from Bio.Seq import Seq
import RNA
from tensorflow.keras.optimizers import Adam


In [ ]:
# Set Reproducible Seed

# Full reproducibility
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)


In [ ]:
# Sequence Embedding Function 

def make_data(X):
    X = [seq[:21] for seq in X]  # Use only first 21 bases
    vectorizer = Tokenizer(lower=False, split=" ", num_words=None, char_level=True)
    vectorizer.fit_on_texts(X)
    alphabet = "ATCG"
    char_dict = {char: i + 1 for i, char in enumerate(alphabet)}
    word_index = {k: (v + 1) for k, v in char_dict.items()}
    word_index["PAD"] = 0
    word_index["START"] = 1
    vectorizer.word_index = word_index
    X = vectorizer.texts_to_sequences(X)
    X = [[word_index["START"]] + x for x in X]
    X = pad_sequences(X, maxlen=22)
    return np.array(X)


In [ ]:
# Biological Feature Extraction

def extract_features(data):
    sequences = data['23mer']
    features = []
    for seq in sequences:
        seq = seq.upper()[:21]
        a_count = seq.count('A') / len(seq)
        t_count = seq.count('T') / len(seq)
        c_count = seq.count('C') / len(seq)
        g_count = seq.count('G') / len(seq)
        gc_content = (g_count + c_count) / len(seq)
        tm_global = Tm.Tm_Wallace(seq)
        tm_pam_proximal = Tm.Tm_Wallace(seq[-8:])
        tm_middle = Tm.Tm_Wallace(seq[7:14])
        ss, mfe = RNA.fold(seq)
        stem_loop = 1 if 'GG' in seq else 0
        free_energy = mfe
        base_accessibility = 1 - gc_content
        features.append([
            a_count, t_count, c_count, g_count,
            gc_content, tm_global, tm_pam_proximal, tm_middle,
            stem_loop, free_energy, base_accessibility
        ])
    feature_names = [
        'A_Content', 'T_Content', 'C_Content', 'G_Content',
        'GC_Content', 'Tm_Global', 'Tm_PAM_Proximal', 'Tm_Middle',
        'Stem_Loop', 'Free_Energy', 'Base_Accessibility'
    ]
    return np.array(features), feature_names


In [ ]:
# DeepHF BiLSTM Model 

def build_model(sequence_length=22, biofeat_dim=11):
    sequence_input = Input(shape=(sequence_length,), name='seq_input')
    embedding = Embedding(input_dim=7, output_dim=128)(sequence_input)
    lstm = Bidirectional(LSTM(256, return_sequences=True))(embedding)
    lstm = Bidirectional(LSTM(128, return_sequences=True))(lstm)
    lstm = Flatten()(lstm)

    biofeat_input = Input(shape=(biofeat_dim,), name='bio_input')
    biofeat = Dense(512, activation='relu')(biofeat_input)
    biofeat = Dense(256, activation='relu')(biofeat)

    concatenated = Concatenate()([lstm, biofeat])
    output = Dense(1, activation='linear')(concatenated)

    model = Model(inputs=[sequence_input, biofeat_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.00001), loss='mse')
    
    return model


In [ ]:
# SHAP values calculation

def calculate_shap_values(model, X_seq, X_biofeat):
    def model_predict(input_data):
        return model.predict([input_data[:, :22], input_data[:, 22:]])
    
    test_data = np.column_stack([X_seq, X_biofeat])
    explainer = shap.Explainer(model_predict, test_data)
    shap_values = explainer(test_data).values  
    return shap_values


In [ ]:
# SHAP Waterfall Plot

def plot_waterfall(shap_values, feature_names, pred_value, mean_value, index):
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[index],
            base_values=mean_value,
            feature_names=feature_names
        )
    )
    plt.title(f'Waterfall Plot for Sample {index+1} (f(x) = {pred_value:.6f}, E(x) = {mean_value:.6f})')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_plot_{index+1}.png')
    plt.close()


In [ ]:
# prediction + SHAP pipline

def predict_efficiencies(df):
    df['21mer'] = df['23mer'].str[:21]
    X_seq = make_data(df['21mer'])
    X_biofeat, feature_names = extract_features(df)
    
    # Normalize biological features
    X_biofeat = (X_biofeat - X_biofeat.mean(axis=0)) / X_biofeat.std(axis=0)
    
    model = build_model()
    model.fit(
        [X_seq, X_biofeat], df['Scores DeepHF'].values,
        epochs=2000,
        batch_size=32,
        verbose=1,
        shuffle=False
    )
    
    model.save('deephf_trained_model.h5')
    print("Trained model saved.")
    
    df['Predicted_Efficiency'] = model.predict([X_seq, X_biofeat]).flatten()
    
    shap_values = calculate_shap_values(model, X_seq, X_biofeat)
    mean_value = df['Predicted_Efficiency'].mean()
    
    for i, feature in enumerate(feature_names):
        df[feature] = X_biofeat[:, i]
        df[f'{feature}_SHAP'] = shap_values[:, -11 + i]

    for i in range(2):  # Plot for first 2 samples
        plot_waterfall(shap_values[:, -11:], feature_names, df['Predicted_Efficiency'][i], mean_value, i)
    
    output_file = 'Predicted_Efficiencies_DeepHF.csv'
    df.to_csv(output_file, index=False)
    print(f" Predictions saved to: {output_file}")
    print("\n Sample Output:")
    print(df[['23mer', 'Scores DeepHF', 'Predicted_Efficiency'] + feature_names + [f'{f}_SHAP' for f in feature_names]])


In [ ]:
# Run the workflow

if __name__ == "__main__":
    input_file = "Prepared_23nt_DeepHF_Input.csv"
    df_input = pd.read_csv(input_file)
    predict_efficiencies(df_input)


In [ ]:
# Finall code

import os
import pandas as pd
import numpy as np
import tensorflow as tf
import random
import shap
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Concatenate, Flatten
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from Bio.SeqUtils import MeltingTemp as Tm
from Bio.Seq import Seq
import RNA
from tensorflow.keras.optimizers import Adam

# --- Set Seeds for Full Reproducibility ---
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# --- Feature Engineering ---
def make_data(X):
    X = [seq[:21] for seq in X]  
    vectorizer = Tokenizer(lower=False, split=" ", num_words=None, char_level=True)
    vectorizer.fit_on_texts(X)
    alphabet = "ATCG"
    char_dict = {char: i + 1 for i, char in enumerate(alphabet)}
    word_index = {k: (v + 1) for k, v in char_dict.items()}
    word_index["PAD"] = 0
    word_index["START"] = 1
    vectorizer.word_index = word_index
    X = vectorizer.texts_to_sequences(X)
    X = [[word_index["START"]] + x for x in X]
    X = pad_sequences(X, maxlen=22)  
    return np.array(X)

def extract_features(data):
    sequences = data['23mer']
    features = []
    for seq in sequences:
        seq = seq.upper()[:21]  
        a_count = seq.count('A') / len(seq)
        t_count = seq.count('T') / len(seq)
        c_count = seq.count('C') / len(seq)
        g_count = seq.count('G') / len(seq)
        gc_content = (seq.count('G') + seq.count('C')) / len(seq)
        tm_global = Tm.Tm_Wallace(seq)
        tm_pam_proximal = Tm.Tm_Wallace(seq[-8:])
        tm_middle = Tm.Tm_Wallace(seq[7:14])
        ss, mfe = RNA.fold(seq)
        stem_loop = 1 if 'GG' in seq else 0
        free_energy = mfe
        base_accessibility = 1 - gc_content
        features.append([
            a_count, t_count, c_count, g_count,
            gc_content, tm_global, tm_pam_proximal, tm_middle,
            stem_loop, free_energy, base_accessibility
        ])
    feature_names = [
        'A_Content', 'T_Content', 'C_Content', 'G_Content',
        'GC_Content', 'Tm_Global', 'Tm_PAM_Proximal', 'Tm_Middle',
        'Stem_Loop', 'Free_Energy', 'Base_Accessibility'
    ]
    return np.array(features), feature_names

# --- Model Architecture ---
def build_model(sequence_length=22, biofeat_dim=11):
    sequence_input = Input(shape=(sequence_length,), name='seq_input')
    embedding = Embedding(input_dim=7, output_dim=128)(sequence_input)
    lstm = Bidirectional(LSTM(256, return_sequences=True))(embedding)
    lstm = Bidirectional(LSTM(128, return_sequences=True))(lstm)
    lstm = Flatten()(lstm)

    biofeat_input = Input(shape=(biofeat_dim,), name='bio_input')
    biofeat = Dense(512, activation='relu')(biofeat_input)
    biofeat = Dense(256, activation='relu')(biofeat)

    concatenated = Concatenate()([lstm, biofeat])
    output = Dense(1, activation='linear')(concatenated)

    model = Model(inputs=[sequence_input, biofeat_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.00001), loss='mse')
    
    return model

# --- SHAP Explainer ---
def calculate_shap_values(model, X_seq, X_biofeat):
    def model_predict(input_data):
        return model.predict([input_data[:, :22], input_data[:, 22:]])
    
    test_data = np.column_stack([X_seq, X_biofeat])
    explainer = shap.Explainer(model_predict, test_data)
    shap_values = explainer(test_data).values  
    return shap_values

# --- Waterfall Plot ---
def plot_waterfall(shap_values, feature_names, pred_value, mean_value, index):
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[index],  
            base_values=mean_value,
            feature_names=feature_names
        )
    )
    plt.title(f'Waterfall Plot for Sample {index+1} (f(x) = {pred_value:.6f}, E(x) = {mean_value:.6f})')
    plt.tight_layout()
    plt.savefig(f'shap_waterfall_plot_{index+1}.png')
    plt.close()

# --- Main Workflow ---
def predict_efficiencies(df):
    df['21mer'] = df['23mer'].str[:21]
    X_seq = make_data(df['21mer'])
    X_biofeat, feature_names = extract_features(df)
    
    X_biofeat = (X_biofeat - X_biofeat.mean(axis=0)) / X_biofeat.std(axis=0)
    
    model = build_model()
    model.fit(
        [X_seq, X_biofeat], df['Scores DeepHF'].values,
        epochs=2000,
        batch_size=32,
        verbose=1,
        shuffle=False
    )
    
    df['Predicted_Efficiency'] = model.predict([X_seq, X_biofeat]).flatten()
    
    shap_values = calculate_shap_values(model, X_seq, X_biofeat)
    mean_value = df['Predicted_Efficiency'].mean()
    
    # Save each engineered feature in individual columns
    for i, feature in enumerate(feature_names):
        df[feature] = X_biofeat[:, i]

    # Save each SHAP value in individual columns
    for i, feature in enumerate(feature_names):
        df[f'{feature}_SHAP'] = shap_values[:, -11 + i]

    for i in range(2):
        plot_waterfall(shap_values[:, -11:], feature_names, df['Predicted_Efficiency'][i], mean_value, i)
    
    output_file = 'Predicted_Efficiencies_Perfect_Match_Final.csv'
    df.to_csv(output_file, index=False)
    print(f"Predictions saved to {output_file}")
    print("\nFinal Results:")
    print(df[['23mer', 'Scores DeepHF', '21mer', 'Predicted_Efficiency'] + feature_names + [f'{feature}_SHAP' for feature in feature_names]])

if __name__ == "__main__":
    input_file = "/home/faiza/vienna/mmmm/Prepared_23nt_DeepHF_Input.csv"
    df_input = pd.read_csv(input_file)
    predict_efficiencies(df_input)
